# Notebook 06: Interactive Portfolio Dashboard

**Project:** GNN-Based Battery Voltage Predictor  

---

## Overview

This notebook assembles all results into a standalone interactive HTML dashboard
for portfolio presentation. The dashboard contains four panels:

1. **Parity plot**: Predicted vs actual voltage (all models, colored by chemistry)
2. **Model comparison**: MAE and RMSE bar chart for RF, CGCNN, M3GNet
3. **Screening map**: Predicted voltage vs estimated capacity for novel candidates,
   with hover labels showing formula and stability
4. **Candidate table**: Top 50 screened candidates with sortable columns

All panels use Plotly for interactivity. The final figure is exported to
`results/dashboard.html` as a self-contained file (no server required).

In [ ]:
import sys
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.evaluate import PALETTE, compute_metrics

DATA_DIR    = project_root / 'data'
MODELS_DIR  = project_root / 'models'
RESULTS_DIR = project_root / 'results'

print('Plotly dashboard builder ready.')


## 1. Load Saved Results

We load predictions saved by Notebook 04 (evaluation) and the top candidates
from Notebook 05 (screening). If running for the first time, run notebooks
04 and 05 first to generate these files.

In [ ]:
import torch
from src.data import VoltageGraphDataset
from src.models import build_cgcnn, CrystalTransformer
from src.train import predict, make_loaders
from src.utils import set_seed

set_seed(42)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

with open(DATA_DIR / 'splits.pkl', 'rb') as f:
    splits = pickle.load(f)
test_entries = splits['test']

with open(DATA_DIR / 'matminer_filtered.pkl', 'rb') as f:
    mm = pickle.load(f)
X_test, y_test_mm = mm['X_test'], mm['y_test']

with open(MODELS_DIR / 'rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

test_ds = VoltageGraphDataset.from_processed(str(DATA_DIR / 'test_graphs.pkl'))
_, _, test_loader = make_loaders(test_ds, test_ds, test_ds, batch_size=128, num_workers=0)

# CGCNN
cgcnn = build_cgcnn(node_dim=9, edge_dim=64, hidden_dim=128, n_conv=4, dropout=0.1)
cgcnn.load_state_dict(torch.load(MODELS_DIR / 'cgcnn_best.pt', map_location=device))
cgcnn = cgcnn.to(device)
cgcnn_pred, cgcnn_true = predict(cgcnn, test_loader, device)

# CrystalTransformer
ct_model = CrystalTransformer(node_dim=9, edge_dim=64, hidden_dim=128,
                               n_conv=4, heads=4, dropout=0.1)
ct_model.load_state_dict(torch.load(MODELS_DIR / 'transformer_best.pt', map_location=device))
ct_model = ct_model.to(device)
ct_pred, ct_true = predict(ct_model, test_loader, device)

test_chemistry = [e['chemistry_family'] for e in test_entries]
n_test = len(cgcnn_pred)

rf_pred = rf_model.predict(X_test)
y_test_rf = y_test_mm

all_metrics = {
    'Random Forest':      compute_metrics(y_test_rf, rf_pred),
    'CGCNN':              compute_metrics(cgcnn_true, cgcnn_pred),
    'CrystalTransformer': compute_metrics(ct_true, ct_pred),
}

top_candidates = pd.read_csv(RESULTS_DIR / 'top_candidates.csv')
screening_all  = pd.read_csv(RESULTS_DIR / 'screening_all.csv')

print(f'Test: {n_test} entries')
print(f'Top candidates: {len(top_candidates)}')
for name, m in all_metrics.items():
    print(f'  {name}: MAE={m["MAE"]:.4f} V, R2={m["R2"]:.4f}')


In [ ]:
# CrystalTransformer is the best model -- used throughout
parity_true  = ct_true
parity_pred  = ct_pred
parity_chem  = test_chemistry[:len(ct_pred)]
parity_model_name = 'CrystalTransformer'


## 2. Build Interactive Dashboard

The dashboard uses Plotly subplots arranged in a 2x2 grid:

- Top left: Parity plot (predicted vs actual) for best model, colored by chemistry
- Top right: Model comparison bar chart (MAE and RMSE)
- Bottom left: Screening scatter (voltage vs capacity for novel candidates)
- Bottom right: Top 20 candidate table

In [ ]:
PLOT_COLORS = {
    'oxide': '#2196F3', 'phosphate': '#4CAF50',
    'sulfide': '#FF9800', 'fluoride': '#9C27B0',
    'sulfate': '#F44336', 'silicate': '#009688', 'other': '#607D8B',
}

# Parity plot traces
parity_fig_data = []
for fam in sorted(set(parity_chem)):
    mask = np.array(parity_chem) == fam
    parity_fig_data.append(go.Scatter(
        x=parity_true[mask].tolist(), y=parity_pred[mask].tolist(),
        mode='markers', name=fam,
        marker=dict(color=PLOT_COLORS.get(fam, '#607D8B'), size=5, opacity=0.6),
    ))
met = all_metrics['CrystalTransformer']
vmin = float(min(parity_true.min(), parity_pred.min())) - 0.2
vmax = float(max(parity_true.max(), parity_pred.max())) + 0.2
parity_fig_data.append(go.Scatter(
    x=[vmin, vmax], y=[vmin, vmax],
    mode='lines', showlegend=False,
    line=dict(color='black', dash='dash', width=1),
))

# Screening scatter: use full screening_all with plausible voltages
df_screen_plot = screening_all[
    (screening_all['predicted_voltage_V'] >= 0) &
    (screening_all['predicted_voltage_V'] <= 7)
].copy()

screen_fig_data = []
for fam in df_screen_plot['chemistry_family'].unique():
    subset = df_screen_plot[df_screen_plot['chemistry_family'] == fam]
    screen_fig_data.append(go.Scatter(
        x=subset['est_capacity_mAh_g'].tolist(),
        y=subset['predicted_voltage_V'].tolist(),
        mode='markers', name=fam,
        marker=dict(color=PLOT_COLORS.get(fam, '#607D8B'), size=4, opacity=0.35),
    ))

# Highlight top 10
top10 = top_candidates.head(10)
screen_fig_data.append(go.Scatter(
    x=top10['est_capacity_mAh_g'].tolist(),
    y=top10['predicted_voltage_V'].tolist(),
    mode='markers+text', name='Top 10',
    text=top10['formula'].tolist(),
    textposition='top center',
    textfont=dict(size=8),
    marker=dict(color='gold', size=10, symbol='star',
                line=dict(color='black', width=0.5)),
))

model_names  = list(all_metrics.keys())
maes  = [all_metrics[m]['MAE']  for m in model_names]
rmses = [all_metrics[m]['RMSE'] for m in model_names]
model_colors = ['#E91E63', '#2196F3', '#4CAF50']
print('Plot data ready.')


In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'{parity_model_name}: Predicted vs Actual Voltage (Test Set)',
        'Model Comparison: MAE and RMSE on Test Set',
        'Novel Li Candidates: Predicted Voltage vs Estimated Capacity',
        'Top 20 Screened Candidates',
    ],
    specs=[
        [{'type': 'scatter'}, {'type': 'bar'}],
        [{'type': 'scatter'}, {'type': 'table'}],
    ],
    column_widths=[0.55, 0.45],
    row_heights=[0.50, 0.50],
    vertical_spacing=0.13,
    horizontal_spacing=0.10,
)

# Panel 1: parity plot
for trace in parity_fig_data:
    fig.add_trace(trace, row=1, col=1)

# Panel 2: model comparison bars (grouped by metric)
for mn, mae, rmse, col in zip(model_names, maes, rmses, model_colors):
    fig.add_trace(go.Bar(
        name=mn, x=['MAE', 'RMSE'],
        y=[mae, rmse],
        marker_color=col, opacity=0.85,
        text=[f'{mae:.3f}', f'{rmse:.3f}'],
        textposition='outside',
        showlegend=False,
    ), row=1, col=2)

# Panel 3: screening scatter
for trace in screen_fig_data:
    trace.showlegend = False
    fig.add_trace(trace, row=2, col=1)
fig.add_hline(y=2.0, line_dash='dot', line_color='grey', opacity=0.5, row=2, col=1)

# Panel 4: Top 20 table
top20 = top_candidates.head(20)
alternating = ['#E3F2FD' if i % 2 == 0 else 'white' for i in range(len(top20))]
fig.add_trace(go.Table(
    header=dict(
        values=['#', 'Formula', 'Family', 'Voltage (V)', 'Cap. (mAh/g)', 'Ehull (meV)'],
        fill_color='#1565C0',
        font=dict(color='white', size=10.5, family='Arial'),
        align='center', height=28,
    ),
    cells=dict(
        values=[
            top20['rank'].tolist(),
            top20['formula'].tolist(),
            top20['chemistry_family'].tolist(),
            top20['predicted_voltage_V'].round(3).tolist(),
            top20['est_capacity_mAh_g'].round(1).tolist(),
            top20['energy_above_hull_meV'].round(1).tolist(),
        ],
        fill_color=[alternating] * 6,
        font=dict(size=9.5, family='Arial'),
        align='center', height=24,
    ),
), row=2, col=2)

# Layout
fig.update_layout(
    title=dict(
        text='GNN Battery Voltage Predictor: Portfolio Dashboard',
        font=dict(size=18, family='Arial', color='#1A237E'),
    ),
    template='plotly_white',
    font=dict(family='Arial, Helvetica, sans-serif', size=11),
    height=900,
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.85)',
                bordercolor='lightgrey', borderwidth=1),
    barmode='group',
)

# Axis labels
fig.update_xaxes(title_text='DFT Voltage (V vs Li/Li+)', row=1, col=1)
fig.update_yaxes(title_text='Predicted Voltage (V)', row=1, col=1)
fig.update_xaxes(title_text='Metric', row=1, col=2)
fig.update_yaxes(title_text='Error (V)', row=1, col=2)
fig.update_xaxes(title_text='Estimated Capacity (mAh/g)', row=2, col=1)
fig.update_yaxes(title_text='Predicted Voltage (V vs Li/Li+)', row=2, col=1)

# Annotation: CrystalTransformer metrics
fig.add_annotation(
    xref='x', yref='y', x=vmin + 0.4, y=vmax - 0.3,
    text=f"MAE = {met['MAE']:.3f} V<br>R<sup>2</sup> = {met['R2']:.3f}",
    showarrow=False, font=dict(size=10, color='#1A237E'),
    bgcolor='rgba(255,255,255,0.85)', bordercolor='lightgrey',
    borderwidth=1, row=1, col=1,
)

print('Dashboard figure built.')


In [ ]:
output_path = RESULTS_DIR / 'dashboard.html'
fig.write_html(
    str(output_path),
    include_plotlyjs='cdn',
    full_html=True,
    config={'displayModeBar': True, 'scrollZoom': True},
)
print(f'Dashboard saved to {output_path}')
print(f'File size: {output_path.stat().st_size / 1024:.1f} KB')


## 3. Additional Figures for Portfolio Page

In [ ]:
with open(DATA_DIR / 'battery_dataset.json') as f:
    all_entries = json.load(f)

df_all = pd.DataFrame([{
    'average_voltage': e['average_voltage'],
    'chemistry_family': e['chemistry_family'],
    'formula': e['formula'],
    'capacity_grav': e['capacity_grav'],
} for e in all_entries])

fig_dist = px.histogram(
    df_all, x='average_voltage',
    color='chemistry_family',
    color_discrete_map=PLOT_COLORS,
    nbins=60,
    title='Li Intercalation Voltage Distribution by Chemistry Family',
    labels={'average_voltage': 'Average Voltage (V vs Li/Li+)', 'count': 'Number of Entries'},
    barmode='overlay', opacity=0.75, template='plotly_white',
)
fig_dist.update_layout(
    font=dict(family='Arial, Helvetica, sans-serif'),
    legend_title='Chemistry Family',
)
fig_dist.write_html(
    str(RESULTS_DIR / 'fig_voltage_distribution_interactive.html'),
    include_plotlyjs='cdn',
)
print('Saved: fig_voltage_distribution_interactive.html')

# Screening interactive figure
fig_screen = px.scatter(
    df_screen_plot,
    x='est_capacity_mAh_g',
    y='predicted_voltage_V',
    color='chemistry_family',
    color_discrete_map=PLOT_COLORS,
    hover_data=['formula', 'material_id', 'energy_above_hull_meV'],
    title='Novel Li Candidate Screening (Interactive)',
    labels={
        'est_capacity_mAh_g': 'Estimated Capacity (mAh/g)',
        'predicted_voltage_V': 'Predicted Voltage (V vs Li/Li+)',
    },
    opacity=0.5,
    template='plotly_white',
)
fig_screen.add_hline(y=2.0, line_dash='dot', line_color='grey', opacity=0.6,
                     annotation_text='2.0 V cutoff')
fig_screen.write_html(
    str(RESULTS_DIR / 'fig_screening_interactive.html'),
    include_plotlyjs='cdn',
)
print('Saved: fig_screening_interactive.html')


In [ ]:
print('Dashboard notebook complete!')
print(f'Outputs in {RESULTS_DIR}:')
for p in sorted(RESULTS_DIR.glob('*.*')):
    size = p.stat().st_size
    print(f'  {p.name:50s} {size/1024:.1f} KB')
